In [3]:
import torch
import torch.nn as nn

# Vocabulary
src_vocab = {
    "<pad>": 0,
    "<sos>": 1,
    "<eos>": 2,
    "hello": 3,
    "thanks": 4,
    "good": 5,
    "morning": 6
}

tgt_vocab = {
    "<pad>": 0,
    "<sos>": 1,
    "<eos>": 2,
    "bonjour": 3,
    "merci": 4,
    "bon": 5,
    "matin": 6
}


class Encoder(nn.Module):
    def __init__(self, input_size, embedding_size,
                 hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(
            input_size,
            embedding_size
        )

        self.rnn = nn.LSTM(
            embedding_size,
            hidden_size,
            bidirectional=True,
            batch_first=True
        )

    def forward(self, x):
        embedded = self.embedding(x)

        output, (hidden, cell) = self.rnn(embedded)

        return output, hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_size,
                 embedding_size,
                 hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(
            output_size,
            embedding_size
        )

        self.rnn = nn.LSTM(
            embedding_size,
            hidden_size * 2,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size * 2,
            output_size
        )

    def forward(self, x, hidden, cell):

        x = x.unsqueeze(1)

        embedded = self.embedding(x)

        output, (hidden, cell) = self.rnn(
            embedded,
            (hidden, cell)
        )

        prediction = self.fc(output.squeeze(1))

        return prediction, hidden, cell


# Model parameters
INPUT_SIZE = len(src_vocab)
OUTPUT_SIZE = len(tgt_vocab)

EMBEDDING_SIZE = 32
HIDDEN_SIZE = 64

encoder = Encoder(
    INPUT_SIZE,
    EMBEDDING_SIZE,
    HIDDEN_SIZE
)

decoder = Decoder(
    OUTPUT_SIZE,
    EMBEDDING_SIZE,
    HIDDEN_SIZE
)

print("Bidirectional RNN Encoder-Decoder")
print("----------------------------------")

print("Input vocabulary size :", INPUT_SIZE)
print("Output vocabulary size:", OUTPUT_SIZE)
print("Embedding size        :", EMBEDDING_SIZE)
print("Hidden size            :", HIDDEN_SIZE)
print("Encoder direction      : Bidirectional")
print("Decoder direction      : Unidirectional")

# Test input
sentence = torch.tensor([[src_vocab["hello"]]])

output, hidden, cell = encoder(sentence)

print("\nEncoder output shape:", output.shape)
print("Encoder hidden shape:", hidden.shape)

Bidirectional RNN Encoder-Decoder
----------------------------------
Input vocabulary size : 7
Output vocabulary size: 7
Embedding size        : 32
Hidden size            : 64
Encoder direction      : Bidirectional
Decoder direction      : Unidirectional

Encoder output shape: torch.Size([1, 1, 128])
Encoder hidden shape: torch.Size([2, 1, 64])
